# Mandelbrot set

This notebook estimates the area of the Mandelbrot set using parallel processing with OpenMP.

### How the Mandelbrot Set Works

1. **Grid Mapping**
   - The grid indices are mapped to complex numbers within a specific range.
   - For each point `c`, we check if iterating the function $z_{n+1} = z_n² + c$ remains bounded.

2. **Iteration Process**
   - Starting with `z = 0`, compute subsequent values using the formula.
   - If at any iteration, the magnitude of `z` exceeds a threshold (here, 2), the point is marked as escaping and not part of the set.

3. **Counting Escaping Points**
   - `numoutside` keeps track of how many points escape within the given iterations.
   - The area estimate is based on the proportion of these points within the grid.


In [1]:
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

1. **Constants Definition**:
   - `NPOINTS = 5000`: The grid size, creating a 5000x5000 grid.
   - `MAXITER = 1000`: Maximum iterations per point to determine if it escapes.

In [2]:
#define NPOINTS 5000
#define MAXITER 1000

int numoutside = 0;

2. Function testpoint:
   - Takes complex number parameters `creal` and `cimag`.
   - Initializes `zreal` and `zimag` to 0.
   - Iterates up to `MAXITER`, checking if `(zreal^2 + zimag^2) > 4.0`. If so, the point escapes, incrementing `numoutside`.

In [3]:
void testpoint(double creal, double cimag) {
    double zreal = 0.0; // Start at 0
    double zimag = 0.0; 
    double r2, i2;
    int iter;

    for (iter = 0; iter < MAXITER; iter++) {
        r2 = zreal * zreal;
        i2 = zimag * zimag;
        
        if ((r2 + i2) > 4.0) {
            #pragma omp atomic // Safety check if not using reduction correctly
            numoutside++;
            return;
        }
        
        zimag = 2.0 * zreal * zimag + cimag;
        zreal = r2 - i2 + creal;
    }
}

3. **Main Function**
   - Maps grid indices to complex numbers within a specific range using scaling factors:
     ```
     creal = (i * (-1.0 / ((double)(NPOINTS - 1)))) + (-1.0);
     cimag = (j * (-1.0 / ((double)(NPOINTS - 1)))) + (-0.75);
     ```
   - Uses `#pragma omp parallel for` to distribute the computation across threads.
   - `default(shared)` means variables not specified are shared among all threads.
   - `private(i, j, creal, cimag)` ensures each thread has its own copy of these variables.
   - `reduction(+:numoutside)` accumulates the total count of escaping points.


3. **Area Calculation**:
   - Computes area as `(2.0 * 2.5 * 1.125) * ((NPOINTS^2 - numoutside) / NPOINTS^2)`
   - Error estimate is derived by dividing area by `NPOINTS`.

In [4]:
int main() {
    int i, j;
    double area, error, eps = 1.0e-5;
    double cimag, creal;
    // Loop over grid of points in the complex plane which contains the Mandelbrot set,
    // testing each point to see whether it is inside or outside the set.

// i, j, creal, and cimag MUST be private so threads don't overwrite each other
#pragma omp parallel for default(shared) private(i, j, creal, cimag) reduction(+:numoutside)
for (i = 0; i < NPOINTS; i++) {
    for (j = 0; j < NPOINTS; j++) {
        creal = -2.0 + 2.5 * (double)(i) / (double)(NPOINTS) + eps;
        cimag = 1.125 * (double)(j) / (double)(NPOINTS) + eps;
        testpoint(creal, cimag);
    }
}

    // Calculate area of set and error estimate and output the results
    area = 2.0 * 2.5 * 1.125 * (double) (NPOINTS * NPOINTS - numoutside) / (double) (NPOINTS * NPOINTS);
    error = area / (double) NPOINTS;

    printf("Area of Mandlebrot set = %12.8f +/- %12.8f\n", area, error);
    printf("Correct answer should be around 1.510659\n");
}

In [5]:
main();

Area of Mandlebrot set =   1.51064145 +/-   0.00030213
Correct answer should be around 1.510659
